# Model and Question Selection

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score, davies_bouldin_score

## Prepare the Behavioral Data

In [ ]:
DATA_PATH = "../data/Dataset_Eating_Disorder.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df.rename(columns={
    "DesireToBuy _FromSnackBarOrCafe": "DesireToBuy_FromSnackBarOrCafe"
})

duplicate_count = df.duplicated().sum()
df_unique = df.drop_duplicates().copy()

print("Original shape:", df.shape)
print("Duplicate rows:", duplicate_count)
print("Unique shape:", df_unique.shape)

In [ ]:
behavioral_groups = {
    "Restrained Eating": [
        "EatLess_OnWeightGain", "EatLess_AtMealtime", "RefuseFood_WeightConcern", "Monitor_Food",
        "Eat_SlimmingFoods", "EatLess_AfterOvereating", "EatLess_ToPreventWeightGain",
        "AvoidSnacks_BetweenMealsToWatchWeight", "AvoidEveningEating_ToWatchWeight",
        "ConsiderWeight_WhenEating"
    ],
    "Emotional Eating": [
        "Eat_WhenIrritated", "Eat_WhenUnoccupied", "Eat_WhenDepressedOrDiscouraged", "Eat_WhenLonely",
        "Eat_WhenSomeoneLetDown", "Eat_WhenAngry", "Eat_WhenExpectingBad", "Eat_WhenAnxious",
        "Eat_WhenThingsGoWrong", "Eat_WhenFrightened", "Eat_WhenDisappointed",
        "Eat_WhenEmotionallyUpset", "Eat_WhenBoredOrRestless"
    ],
    "External / Food-cue Eating": [
        "EatMore_IfFoodTasty", "EatMore_IfFoodSmellsOrLooksGood", "Eat_WhenSeeDeliciousFood",
        "Eat_DeliciousFoodImmediately", "DesireToBuy_FromBakery", "DesireToBuy_FromSnackBarOrCafe",
        "DesireToEat_WhenSeeOthersEating", "Resist_DeliciousFood", "EatMore_WhenSeeOthersEating",
        "Eat_WhenPreparingMeal"
    ],
    "Body Image / Shape Concern": [
        "Days_DesireForFlatStomach", "Days_FeltFat", "Days_WeightAffectedSelfJudgment",
        "Days_ShapeAffectedSelfJudgment", "Days_DissatisfiedWithWeight", "Days_DissatisfiedWithShape",
        "Days_UncomfortableToSeeOwnBody", "Days_UncomfortableBecauseOthersSeeingShape",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FastedControlShapeOrWeight",
        "Days_ExcludedFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FearLosingControlOverEating"
    ],
    "Habitual Eating": [
        "Eat_SpecificFoodsHabitually", "Location_TriggersHabitualEating",
        "AutomaticEating_WhenExperiencingStrongEmotion", "Realize_AfterEatingOutOfHabit"
    ]
}

behavioral_features = [question for questions in behavioral_groups.values() for question in questions]
feature_domain = {
    question: domain
    for domain, questions in behavioral_groups.items()
    for question in questions
}

print("Behavioral questions:", len(behavioral_features))

In [ ]:
scale_a_features = [question for question in behavioral_features if not question.startswith("Days_")]
scale_b_features = [question for question in behavioral_features if question.startswith("Days_")]

scale_a_mapping = {
    "Never": 0, "Seldom": 1, "Sometimes": 2, "Often": 3, "Very often": 4
}
scale_b_mapping = {
    "No days": 0, "1-5 days": 1, "6-12 days": 2, "13-15 days": 3, "Every day": 4
}

df_processed = df_unique.copy()

for column in scale_a_features:
    df_processed[column] = df_processed[column].map(scale_a_mapping)

for column in scale_b_features:
    df_processed[column] = df_processed[column].map(scale_b_mapping)

behavioral_data = df_processed[behavioral_features].copy()

assert behavioral_data.shape == (601, 50)
assert behavioral_data.isna().sum().sum() == 0
assert behavioral_data.min().min() == 0 and behavioral_data.max().max() == 4

print("Behavioral matrix:", behavioral_data.shape)
print("Missing values:", behavioral_data.isna().sum().sum())
print("Encoded range:", behavioral_data.min().min(), "to", behavioral_data.max().max())

## Initial Cluster Count Evaluation on the Full Questionnaire

In [ ]:
full_k_rows = []

for k in range(2, 6):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(behavioral_data)
    full_k_rows.append({
        "k": k,
        "Silhouette Score": silhouette_score(behavioral_data, labels),
        "Davies-Bouldin Score": davies_bouldin_score(behavioral_data, labels),
        "Cluster Sizes": list(np.bincount(labels))
    })

full_questionnaire_k_results = pd.DataFrame(full_k_rows)
full_questionnaire_k_results.round(3)

## Reference Behavioral Structure

In [ ]:
reference_model = KMeans(n_clusters=2, random_state=42, n_init=10)
reference_labels = reference_model.fit_predict(behavioral_data)

print("Reference cluster sizes:", np.bincount(reference_labels))
print("Reference silhouette:", round(silhouette_score(behavioral_data, reference_labels), 4))

## Simplified Question Evidence

In [ ]:
correlation_50 = behavioral_data.corr(method="spearman")
question_rows = []

for question in behavioral_features:
    group_0 = behavioral_data.loc[reference_labels == 0, question]
    group_1 = behavioral_data.loc[reference_labels == 1, question]
    u_statistic = mannwhitneyu(group_0, group_1).statistic
    effect_size = abs(1 - 2 * u_statistic / (len(group_0) * len(group_1)))
    other_correlations = correlation_50[question].drop(question).abs()

    question_rows.append({
        "Question Code": question,
        "Behavioral Domain": feature_domain[question],
        "Variance": behavioral_data[question].var(),
        "Max |Spearman rho|": other_correlations.max(),
        "Cluster Separation": effect_size
    })

question_evidence = pd.DataFrame(question_rows).set_index("Question Code")
question_evidence.sort_values("Cluster Separation", ascending=False).head(15).round(3)

## Why Approximately 10 Questions?

In [ ]:
candidate_sets = {
    8: [
        "Days_FearLosingControlOverEating", "Days_ExcludedFoodControlShapeOrWeight",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "EatLess_ToPreventWeightGain", "EatLess_AfterOvereating",
        "AvoidEveningEating_ToWatchWeight", "Eat_WhenAnxious"
    ],
    10: [
        "Days_FearLosingControlOverEating", "Days_ExcludedFoodControlShapeOrWeight",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FeltFat", "EatLess_ToPreventWeightGain", "EatLess_AfterOvereating",
        "AvoidEveningEating_ToWatchWeight", "Eat_WhenAnxious", "Eat_WhenThingsGoWrong"
    ],
    12: [
        "Days_FearLosingControlOverEating", "Days_ExcludedFoodControlShapeOrWeight",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FeltFat", "Days_DissatisfiedWithShape", "EatLess_ToPreventWeightGain",
        "EatLess_AfterOvereating", "AvoidEveningEating_ToWatchWeight", "Eat_WhenAnxious",
        "Eat_WhenThingsGoWrong", "RefuseFood_WeightConcern"
    ],
    15: [
        "Days_FearLosingControlOverEating", "Days_ExcludedFoodControlShapeOrWeight",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FeltFat", "Days_DissatisfiedWithShape", "Days_ShapeAffectedSelfJudgment",
        "Days_DissatisfiedWithWeight", "EatLess_ToPreventWeightGain", "EatLess_AfterOvereating",
        "AvoidEveningEating_ToWatchWeight", "Eat_WhenAnxious", "Eat_WhenThingsGoWrong",
        "RefuseFood_WeightConcern", "Eat_SpecificFoodsHabitually"
    ],
    20: [
        "Days_FearLosingControlOverEating", "Days_ExcludedFoodControlShapeOrWeight",
        "Days_TriedLimitFoodControlShapeOrWeight", "Days_FollowedRulesControlShapeOrWeight",
        "Days_FeltFat", "Days_DissatisfiedWithShape", "Days_ShapeAffectedSelfJudgment",
        "Days_DissatisfiedWithWeight", "Days_UncomfortableBecauseOthersSeeingShape",
        "Days_WeightAffectedSelfJudgment", "EatLess_ToPreventWeightGain", "EatLess_AfterOvereating",
        "AvoidEveningEating_ToWatchWeight", "Eat_WhenAnxious", "Eat_WhenThingsGoWrong",
        "RefuseFood_WeightConcern", "Eat_SpecificFoodsHabitually", "Eat_WhenEmotionallyUpset",
        "Eat_WhenBoredOrRestless", "AvoidSnacks_BetweenMealsToWatchWeight"
    ]
}

assert {size: len(questions) for size, questions in candidate_sets.items()} == {
    8: 8, 10: 10, 12: 12, 15: 15, 20: 20
}
print("Candidate sizes:", list(candidate_sets))

In [ ]:
size_rows = []

for size, questions in candidate_sets.items():
    labels = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(
        behavioral_data[questions]
    )
    size_rows.append({
        "Number of Questions": size,
        "ARI": adjusted_rand_score(reference_labels, labels)
    })

size_results = pd.DataFrame(size_rows)
size_results.round(3)

In [ ]:
plt.plot(size_results["Number of Questions"], size_results["ARI"], marker="o")
plt.xticks(size_results["Number of Questions"])
plt.ylim(0, 1)
plt.xlabel("Number of questions")
plt.ylabel("ARI with 50-question reference")
plt.title("Questionnaire Length and Preserved Behavioral Structure")
plt.grid(alpha=0.3)
plt.show()

## Final 10 Questions

In [ ]:
final_questions = candidate_sets[10]

final_question_table = pd.DataFrame({
    "Question Code": final_questions,
    "Behavioral Domain": [feature_domain[question] for question in final_questions]
})

print("Domain composition:")
print(final_question_table["Behavioral Domain"].value_counts())
final_question_table

In [ ]:
final_correlation = behavioral_data[final_questions].corr(method="spearman")
correlated_pairs = []

for i in range(len(final_questions)):
    for j in range(i + 1, len(final_questions)):
        correlated_pairs.append({
            "Question 1": final_questions[i],
            "Question 2": final_questions[j],
            "Spearman rho": final_correlation.iloc[i, j]
        })

strongest_pairs = pd.DataFrame(correlated_pairs)
strongest_pairs["Absolute rho"] = strongest_pairs["Spearman rho"].abs()
strongest_pairs.sort_values("Absolute rho", ascending=False).head(5).round(3)

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(final_correlation, vmin=-1, vmax=1, cmap="coolwarm")
plt.colorbar(label="Spearman rho")
plt.xticks(range(10), range(1, 11))
plt.yticks(range(10), range(1, 11))
plt.xlabel("Question number in final table")
plt.ylabel("Question number in final table")
plt.title("Spearman Correlation of the Final 10 Questions")
plt.tight_layout()
plt.show()

## Confirm the Number of Clusters After Reduction

In [ ]:
cluster_count_rows = []

for k in range(2, 6):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(
        behavioral_data[final_questions]
    )
    cluster_count_rows.append({
        "k": k,
        "Silhouette Score": silhouette_score(behavioral_data[final_questions], labels),
        "Davies-Bouldin Score": davies_bouldin_score(behavioral_data[final_questions], labels),
        "Cluster Sizes": list(np.bincount(labels))
    })

cluster_count_results = pd.DataFrame(cluster_count_rows)
cluster_count_results.round(3)

In [ ]:
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()

ax1.plot(cluster_count_results["k"], cluster_count_results["Silhouette Score"], marker="o", color="steelblue")
ax2.plot(cluster_count_results["k"], cluster_count_results["Davies-Bouldin Score"], marker="s", color="darkorange")

ax1.set_xticks(cluster_count_results["k"])
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Silhouette (higher is better)", color="steelblue")
ax2.set_ylabel("Davies-Bouldin (lower is better)", color="darkorange")
plt.title("Cluster-count Evaluation on the Final 10 Questions")
plt.show()

## Simplified Algorithm Comparison

In [ ]:
X_final = behavioral_data[final_questions]

algorithm_labels = {
    "K-Means": KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_final),
    "Agglomerative (Ward)": AgglomerativeClustering(n_clusters=2, linkage="ward").fit_predict(X_final),
    "Agglomerative (Complete)": AgglomerativeClustering(n_clusters=2, linkage="complete").fit_predict(X_final)
}

algorithm_rows = []
for algorithm, labels in algorithm_labels.items():
    algorithm_rows.append({
        "Algorithm": algorithm,
        "Silhouette Score": silhouette_score(X_final, labels),
        "Davies-Bouldin Score": davies_bouldin_score(X_final, labels),
        "ARI with 50-question reference": adjusted_rand_score(reference_labels, labels),
        "Cluster Sizes": list(np.bincount(labels))
    })

algorithm_results = pd.DataFrame(algorithm_rows)
algorithm_results.round(3)

## Final Research Decision

#### Final questionnaire size: 10

#### Final questions

1. `Days_FearLosingControlOverEating`
2. `Days_ExcludedFoodControlShapeOrWeight`
3. `Days_TriedLimitFoodControlShapeOrWeight`
4. `Days_FollowedRulesControlShapeOrWeight`
5. `Days_FeltFat`
6. `EatLess_ToPreventWeightGain`
7. `EatLess_AfterOvereating`
8. `AvoidEveningEating_ToWatchWeight`
9. `Eat_WhenAnxious`
10. `Eat_WhenThingsGoWrong`

#### Final number of clusters: 2

#### Final algorithm: K-Means